# 06 · Implantação

**CRISP-DM fase 6, implantação** · Turismo e Hotelaria

---

Atribui um segmento a uma reserva nova em três passos:
1. **modelo:** lê o modelo vencedor do [`04-0-modelagem-cluster`](04-0-modelagem-cluster.ipynb) (K-Means, k=5);
2. **entrada:** a reserva nova, em unidades de negócio (dias de antecedência, tarifa, país...), passa pelo mesmo pré-processamento do [`03-preparacao`](03-preparacao.ipynb);
3. **saída:** o segmento da reserva, com o perfil desse segmento na carteira (taxa de cancelamento, antecedência mediana...), para orientar overbooking, canal e confirmação.

!!! info "Como usar"
    Para classificar reservas novas, edite a lista em `novas_reservas` (seção 4) e
    rode o notebook do início ao fim. Para um lote via CSV, chame
    `segmentar(pd.read_csv('arquivo.csv', sep=';').to_dict('records'))` na seção 5.
    O notebook não precisa de nenhum argumento de linha de comando nem de variável
    de ambiente: os caminhos são resolvidos automaticamente a partir de onde ele
    é executado.

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.spatial.distance import cdist
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DADOS = RAIZ / "data" / "processed"
MODELOS = RAIZ / "models"

## 1 · Modelo treinado

In [ ]:
# vencedor e parâmetros, salvos pelo 04-0
selecionado = json.loads((MODELOS / "selecionado.json").read_text())

try:
    modelo = joblib.load(MODELOS / selecionado["arquivo"])  # modelo treinado
# modelos escritos à mão no notebook (K-Medianas) só carregam onde a classe foi definida
except Exception as erro:
    modelo = None
    print(f"Modelo não carregado ({type(erro).__name__}): atribuição usa o centroide mais próximo")

# Ward e DBSCAN não classificariam reservas novas sozinhos
classifica_novas = hasattr(modelo, "predict")
atribuicao = "predict do modelo" if classifica_novas else "centroide mais próximo"
display(
    Markdown(
        f"Modelo: **{selecionado['modelo']}** · `{selecionado['params']}` · "
        f"{selecionado['grupos']} segmentos · atribuição por **{atribuicao}**"
    )
)

## 2 · Pré-processamento do 03

O 03 salvou a base transformada, mas não o pré-processador. Aqui ele é refeito com as
mesmas regras (mesma lista de colunas, mesmo agrupamento de `country` em 16
categorias) e ajustado na mesma base tratada, então as escalas são as mesmas. A
célula confere que o resultado bate com a saída do 03 antes de seguir.

`is_canceled`, `reservation_status`, `reservation_status_date`, `arrival_date`,
`agent` e `company` ficam fora, como no 04-0: o cancelamento de uma reserva nova nem
sempre é conhecido, e os demais são vazamento ou identificadores de alta cardinalidade.

In [ ]:
# base tratada: referência de escalas e categorias
bruta = pd.read_csv(DADOS / "hotel_bookings_tratado.csv", sep=";")
dicionario = pd.read_csv(DADOS / "dicionario.csv", sep=";")  # tipo de cada variável

USELESS_COLUMNS = [
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "arrival_date",
    "agent",
    "company",
]

TOPO_PAISES = 15
frequencia_paises = bruta["country"].value_counts()
paises_mantidos = frequencia_paises.head(TOPO_PAISES).index

bruta_modelo = bruta.drop(columns=USELESS_COLUMNS, errors="ignore").copy()
bruta_modelo["country"] = np.where(
    bruta_modelo["country"].isin(paises_mantidos), bruta_modelo["country"], "Outro"
)  # mesmo agrupamento do 03


def colunas_do_tipo(*tipos):
    colunas = dicionario.loc[dicionario["Tipo"].isin(tipos), "variavel"]
    return [c for c in colunas if c in bruta.columns and c not in USELESS_COLUMNS]


continuous_columns = colunas_do_tipo("Contínua", "Discreta", "Binária")
nominal_columns = colunas_do_tipo("Nominal")

derived_continuous = ["total_nights", "total_guests"]
derived_binary = ["room_changed", "has_special_requests"]
continuous_columns = list(dict.fromkeys(continuous_columns + derived_continuous + derived_binary))

preprocessador = (
    ColumnTransformer(
        transformers=[
            ("continuous", Pipeline([("normalization", MinMaxScaler())]), continuous_columns),
            (
                "nominal",
                Pipeline(
                    [("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
                ),
                nominal_columns,
            ),
        ],
        verbose_feature_names_out=False,
    )
    .set_output(transform="pandas")
    .fit(bruta_modelo[continuous_columns + nominal_columns])
)

# carteira no espaço do modelo
X_base = preprocessador.transform(bruta_modelo[continuous_columns + nominal_columns])
saida_03 = pd.read_csv(DADOS / "hotel_bookings_preprocessed.csv", sep=";")
assert list(X_base.columns) == list(saida_03.columns) and np.allclose(
    X_base, saida_03
), "o pré-processamento não reproduz o 03"
print(f"Pré-processamento confere com o 03: {X_base.shape[1]} features")

In [ ]:
# rótulos da carteira (04-0)
segmentos = pd.read_csv(DADOS / "segmentos.csv", sep=";")["segmento"].to_numpy()
validos = segmentos >= 0

ordem = (
    pd.Series(bruta["is_canceled"].to_numpy()[validos], index=segmentos[validos])
    .groupby(level=0)
    .mean()
    .sort_values(ascending=False)
    .index
)  # do maior risco de cancelamento ao menor
# mesmos nomes do 05
nome = {s: f"Segmento {chr(65 + i)}" for i, s in enumerate(ordem)} | {-1: "Sem segmento"}
# centro de cada segmento, para o caso sem predict
centroides = X_base[validos].groupby(segmentos[validos]).mean()

perfil = (
    bruta.assign(segmento=pd.Series(segmentos).map(nome))
    .groupby("segmento")
    .agg(
        reservas=("is_canceled", "size"),
        taxa_cancelamento=("is_canceled", "mean"),
        lead_time_mediano=("lead_time", "median"),
        noites_mediana=("total_nights", "median"),
        adr_mediano=("adr", "median"),
    )
    .sort_values("taxa_cancelamento", ascending=False)
)
if classifica_novas:  # o modelo carregado reproduz a carteira?
    concordancia = (modelo.predict(X_base.to_numpy()) == segmentos).mean()
    print(f"Concordância do modelo com segmentos.csv: {concordancia:.1%}")
perfil.style.format(
    {
        "taxa_cancelamento": "{:.1%}",
        "lead_time_mediano": "{:.0f}",
        "noites_mediana": "{:.0f}",
        "adr_mediano": "{:.2f}",
    }
)

## 3 · Função de atribuição

Cada reserva é um dicionário com as colunas da base tratada. **Só é preciso
informar o que se sabe no momento da reserva.** O resto vem da **reserva mais
parecida da carteira**: a que tem as mesmas categorias informadas e os valores
numéricos mais próximos.

Por que não usar o valor mais comum de cada coluna? Porque o canal e o tipo de
quarto mais comuns são os mesmos para qualquer reserva. Como o One-Hot pesa muito na
distância, reservas opostas acabariam no mesmo segmento.

Um país fora dos 15 mais frequentes é tratado como `Outro`, igual ao que o 03 fez na
carteira inteira. Colunas ou categorias que não existem na base geram erro com as
opções válidas, em vez de um segmento errado silencioso.

In [ ]:
COLUNAS_ENTRADA = continuous_columns + nominal_columns
numerica = {c: pd.api.types.is_numeric_dtype(bruta_modelo[c]) for c in COLUNAS_ENTRADA}
maximos = bruta_modelo[COLUNAS_ENTRADA].max(numeric_only=True)
minimos = bruta_modelo[COLUNAS_ENTRADA].min(numeric_only=True)
amplitude = (maximos - minimos).replace(0, 1)  # para comparar colunas em escalas diferentes


def mais_parecida(reserva):
    # Reserva da carteira com as mesmas categorias informadas e os números mais próximos.
    candidatas = bruta_modelo
    for col, valor in reserva.items():
        if not numerica[col]:
            candidatas = candidatas[candidatas[col] == valor]  # mesma categoria
    if candidatas.empty:
        raise ValueError(f"Nenhuma reserva da carteira combina estas categorias: {reserva}")
    informadas = [c for c in reserva if numerica[c]]
    if not informadas:
        return candidatas.iloc[0]
    valores_informados = pd.Series(reserva)[informadas].astype(float)
    distancia = ((candidatas[informadas] - valores_informados).abs() / amplitude[informadas]).sum(
        axis=1
    )
    return candidatas.loc[distancia.idxmin()]  # menor diferença somada


def _bucketizar_country(reserva):
    if "country" not in reserva:
        return reserva
    valido = reserva["country"] in paises_mantidos
    return {**reserva, "country": reserva["country"] if valido else "Outro"}


def segmentar(reservas):
    desconhecidas = set().union(*reservas) - set(COLUNAS_ENTRADA)
    if desconhecidas:
        raise ValueError(f"Colunas que não existem na base tratada: {sorted(desconhecidas)}")

    # país fora dos 15 mais frequentes vira 'Outro', como no 03
    reservas = [_bucketizar_country(reserva) for reserva in reservas]

    for reserva in reservas:  # categoria desconhecida viraria zeros no one-hot
        for col, valor in reserva.items():
            if not numerica[col] and valor not in set(bruta_modelo[col]):
                validos_col = sorted(map(str, bruta_modelo[col].unique()))
                raise ValueError(f'{col}: "{valor}" não existe na base. Válidos: {validos_col}')

    completas = [{**mais_parecida(reserva), **reserva} for reserva in reservas]
    entrada = pd.DataFrame(completas)[COLUNAS_ENTRADA]
    entrada = entrada.astype(bruta_modelo[COLUNAS_ENTRADA].dtypes.to_dict())  # mesmos tipos da base

    X_nova = preprocessador.transform(entrada).to_numpy()  # mesmo espaço do modelo
    if classifica_novas:
        rotulos = modelo.predict(X_nova)  # o próprio modelo decide
    else:
        # segmento de centro mais próximo
        rotulos = centroides.index[cdist(X_nova, centroides).argmin(axis=1)]
    return entrada.assign(segmento=[nome[r] for r in rotulos])

## 4 · Entrada: novas reservas

Edite a lista abaixo e rode as duas células seguintes. Para um lote, use
`segmentar(pd.read_csv('arquivo.csv', sep=';').to_dict('records'))`.

As colunas derivadas (`total_nights`, `total_guests`, `room_changed`,
`has_special_requests`) devem ser coerentes com as originais. Por exemplo,
`has_special_requests = 1` junto com `total_of_special_requests >= 1`.

In [ ]:
novas_reservas = [
    {  # reserva de grupo via agência, com muita antecedência e sem reembolso
        "hotel": "City Hotel",
        "country": "PRT",
        "deposit_type": "Non Refund",
        "market_segment": "Groups",
        "lead_time": 250,
        "total_nights": 3,
        "adr": 90.0,
    },
    {  # reserva direta, de última hora, sem depósito
        "hotel": "Resort Hotel",
        "country": "GBR",
        "deposit_type": "No Deposit",
        "market_segment": "Direct",
        "lead_time": 5,
        "total_nights": 4,
        "adr": 120.0,
    },
]

## 5 · Saída: segmento de cada reserva

In [ ]:
resultado = segmentar(novas_reservas)
# só as colunas que foram digitadas
informadas = sorted(set().union(*novas_reservas), key=COLUNAS_ENTRADA.index)
display(resultado[informadas + ["segmento"]])

for i, linha in resultado.iterrows():  # perfil do segmento atribuído
    p = perfil.loc[linha["segmento"]]
    texto = (
        f"**Reserva {i + 1} -> {linha['segmento']}** · {p['reservas']:.0f} reservas na carteira · "
        f"taxa de cancelamento **{p['taxa_cancelamento']:.1%}** · "
        f"antecedência mediana {p['lead_time_mediano']:.0f} dias · "
        f"tarifa mediana ${p['adr_mediano']:.2f}"
    )
    display(Markdown(texto))

## Como usar este resultado na operação

* **Segmento de alto risco** (maior taxa de cancelamento): política de overbooking
  mais agressiva é defensável, já que uma fração grande dessas reservas não vai se
  confirmar. Se o depósito for `Non Refund`, considerar cobrança antecipada ou
  confirmação adicional antes de liberar o quarto a outra reserva.
* **Segmento de baixo risco** (menor taxa de cancelamento, tipicamente antecedência
  curta): overbooking conservador, porque a reserva tende a se confirmar.
* **Canal e antecedência:** a tabela `perfil` acima, recarregada a cada re-treino,
  diz qual canal e qual faixa de antecedência dominam cada segmento, informação que
  entra na política comercial por segmento, não só na política de overbooking.